# ECAAL Deep Analysis Notebook
Notebook này thực hiện các phân tích chuyên sâu cho mô hình ECAAL (Exp_B: ResNet ASL và Exp_C: EfficientNet CBAM ASL).

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
import os, sys
from pathlib import Path
REPO_URL = 'https://github.com/Thinh59/ECAAL.git'
REPO_DIR = Path('/kaggle/working/ECAAL')
if REPO_DIR.exists():
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
sys.path.insert(0, str(REPO_DIR / 'src'))

In [ ]:
import shutil, json
OUTPUTS_DIR = Path('/kaggle/working/outputs')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = Path('/kaggle/input/datasets/thinhha59/models')
RESULTS_DIR  = DATASET_ROOT / 'results'
SRC_SUBSET   = RESULTS_DIR / 'data' / 'coco_subset'
EXPS = ['exp_B_resnet_asl', 'exp_C_efficientnet_cbam_asl']
for exp in EXPS:
    src = RESULTS_DIR / exp
    dst = OUTPUTS_DIR / exp
    dst.mkdir(parents=True, exist_ok=True)
    for fname in ['best.pth', 'log.json']:
        if (src / fname).exists(): shutil.copy2(src / fname, dst / fname)
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
for fname in ['subset_train_ids.json', 'subset_val_ids.json', 'subset_test_ids.json']:
    if (SRC_SUBSET / fname).exists(): shutil.copy2(SRC_SUBSET / fname, SUBSET_DIR / fname)

In [ ]:
coco_root = '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017'
if os.path.exists(coco_root): print('COCO 2017 dataset found')
else: print('Please add dataset awsaf49/coco-2017-dataset')

In [ ]:
from dataset import COCOMultiLabelDataset, get_train_transform, get_val_transform
from torch.utils.data import DataLoader
import json

train_ids_file = SUBSET_DIR / 'subset_train_ids.json'
test_ids_file = SUBSET_DIR / 'subset_test_ids.json'
train_ids = json.load(open(train_ids_file)) if train_ids_file.exists() else None
test_ids = json.load(open(test_ids_file)) if test_ids_file.exists() else None

train_ds = COCOMultiLabelDataset(coco_root, split='train', transform=get_train_transform(224), subset_ids=train_ids)
test_ds = COCOMultiLabelDataset(coco_root, split='val', transform=get_val_transform(224), subset_ids=test_ids)

train_loader = DataLoader(train_ds, batch_size=32, num_workers=2, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, num_workers=2, shuffle=False)

In [ ]:
from models import build_model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg_B = {'backbone': 'resnet50', 'use_cbam': False, 'num_classes': 80}
cfg_C = {'backbone': 'efficientnet_b0', 'use_cbam': True, 'num_classes': 80}
model_B = build_model(cfg_B).to(device)
model_C = build_model(cfg_C).to(device)
if (OUTPUTS_DIR / 'exp_B_resnet_asl' / 'best.pth').exists():
    model_B.load_state_dict(torch.load(OUTPUTS_DIR / 'exp_B_resnet_asl' / 'best.pth', map_location=device, weights_only=True)['model_state_dict'])
if (OUTPUTS_DIR / 'exp_C_efficientnet_cbam_asl' / 'best.pth').exists():
    model_C.load_state_dict(torch.load(OUTPUTS_DIR / 'exp_C_efficientnet_cbam_asl' / 'best.pth', map_location=device, weights_only=True)['model_state_dict'])
model_B.eval()
model_C.eval()
print('Models loaded.')

In [ ]:
features_B, features_C = {}, {}
def get_features(name, feat_dict):
    def hook(model, input, output):
        feat_dict[name] = output.detach().cpu()
    return hook
model_B.gap.register_forward_hook(get_features('feats', features_B))
model_C.gap.register_forward_hook(get_features('feats', features_C))

In [ ]:
import numpy as np
from tqdm.auto import tqdm
def run_inference(loader):
    all_targets, all_preds_B, all_preds_C, all_feats_B, all_feats_C = [], [], [], [], []
    with torch.no_grad():
        for images, targets in tqdm(loader):
            if isinstance(targets, tuple): targets = targets[0]
            images = images.to(device)
            out_B = torch.sigmoid(model_B(images)).cpu()
            out_C = torch.sigmoid(model_C(images)).cpu()
            all_targets.append(targets.cpu())
            all_preds_B.append(out_B)
            all_preds_C.append(out_C)
            all_feats_B.append(features_B['feats'].flatten(1))
            all_feats_C.append(features_C['feats'].flatten(1))
    return {'targets': torch.cat(all_targets).numpy(), 'preds_B': torch.cat(all_preds_B).numpy(), 'preds_C': torch.cat(all_preds_C).numpy(), 'feats_B': torch.cat(all_feats_B).numpy(), 'feats_C': torch.cat(all_feats_C).numpy()}
print('Running inference on test set...')
test_results = run_inference(test_loader)
print('Running inference on train set...')
train_results = run_inference(train_loader)

## 1. Vẽ Confusion Matrix
Phân tích số nhãn đoán đúng/sai trên tập Train và Test (Model C).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import multilabel_confusion_matrix
def plot_confusion_matrix(results, model_name, dataset_name, threshold=0.5):
    preds = (results[f'preds_{model_name}'] > threshold).astype(int)
    targets = results['targets'].astype(int)
    mcm = multilabel_confusion_matrix(targets, preds)
    tn, fp, fn, tp = mcm[:, 0, 0].sum(), mcm[:, 0, 1].sum(), mcm[:, 1, 0].sum(), mcm[:, 1, 1].sum()
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'Global Confusion Matrix - {dataset_name} (Model {model_name})')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.xaxis.set_ticklabels(['Negative', 'Positive'])
    ax.yaxis.set_ticklabels(['Negative', 'Positive'])
    plt.show()
    errors_per_class = mcm[:, 0, 1] + mcm[:, 1, 0]
    top_err_idx = np.argsort(errors_per_class)[::-1][:5]
    print(f"Top 5 classes with most errors in {dataset_name}:")
    for idx in top_err_idx:
        print(f"Class {idx}: TP={mcm[idx,1,1]}, TN={mcm[idx,0,0]}, FP={mcm[idx,0,1]}, FN={mcm[idx,1,0]}")
plot_confusion_matrix(train_results, 'C', 'Train')
plot_confusion_matrix(test_results, 'C', 'Test')

## 2. Kiểm tra dữ liệu mất cân bằng
Xem xét phân bố lớp Positive và Negative và so sánh với tần suất mô hình dự đoán.

In [ ]:
def analyze_imbalance(results, dataset_name, model_name='C', threshold=0.5):
    targets = results['targets']
    preds = (results[f'preds_{model_name}'] > threshold).astype(int)
    pos_counts = targets.sum(axis=0)
    pred_pos_counts = preds.sum(axis=0)
    fig, ax = plt.subplots(figsize=(15, 5))
    x = np.arange(80)
    width = 0.35
    ax.bar(x - width/2, pos_counts, width, label='True Positives')
    ax.bar(x + width/2, pred_pos_counts, width, label='Predicted Positives')
    ax.set_yscale('log')
    ax.set_title(f'Data Imbalance & Predictions - {dataset_name}')
    ax.set_xlabel('Class ID')
    ax.set_ylabel('Count (Log Scale)')
    ax.legend()
    plt.show()
    print(f"{dataset_name} Imbalance:\n- Max class pos: {pos_counts.max()}\n- Min class pos: {pos_counts.min()}\n- Avg pos per class: {pos_counts.mean():.2f}")
analyze_imbalance(train_results, 'Train')
analyze_imbalance(test_results, 'Test')
print("\nGiải pháp: Sự mất cân bằng giữa các lớp rất lớn (Long-tailed distribution).")
print("Asymmetric Loss (ASL) đã được sử dụng trong Exp_C để giảm thiểu tác động của Negative labels áp đảo.")
print("Ngoài ra có thể dùng Resampling, K-Fold CV, hoặc Class Weights.")

## 3. Visualize Feature Vectors (Train vs Test)
Xem phần giao (overlap) giữa các vector đặc trưng của tập Train và Test bằng PCA.

In [ ]:
from sklearn.decomposition import PCA
def visualize_features():
    print('Extracting features for PCA visualization...')
    np.random.seed(42)
    train_idx = np.random.choice(len(train_results['feats_C']), min(2000, len(train_results['feats_C'])), replace=False)
    test_idx = np.random.choice(len(test_results['feats_C']), min(2000, len(test_results['feats_C'])), replace=False)
    X = np.concatenate([train_results['feats_C'][train_idx], test_results['feats_C'][test_idx]], axis=0)
    labels = np.array(['Train'] * len(train_idx) + ['Test'] * len(test_idx))
    X_pca = PCA(n_components=2).fit_transform(X)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=labels, alpha=0.5, s=15)
    plt.title('PCA of Extracted Features (Train vs Test)')
    plt.show()
    print("Nhận xét: Nếu overlap lớn, tập Train và Test có phân phối đặc trưng tương đồng.")
visualize_features()

## 4. Test Sample Analysis (Novelty vs Accuracy)
Phân tích xem các mẫu test khác với train (khoảng cách lớn) thì có dễ bị đoán sai không?

In [ ]:
from sklearn.metrics.pairwise import cosine_distances
def analyze_test_samples(threshold=0.5):
    print('Analyzing Test Samples vs Train Samples...')
    test_feats = test_results['feats_C'][:100] # Phân tích 100 mẫu test đầu tiên
    train_feats = train_results['feats_C']
    min_dists = np.min(cosine_distances(test_feats, train_feats), axis=1)
    test_preds = (test_results['preds_C'][:100] > threshold).astype(int)
    test_targets = test_results['targets'][:100].astype(int)
    errors = np.sum(test_preds != test_targets, axis=1)
    plt.figure(figsize=(8, 6))
    plt.scatter(min_dists, errors, alpha=0.6)
    plt.xlabel('Distance to nearest Train sample')
    plt.ylabel('Number of label prediction errors')
    plt.title('Prediction Errors vs. Novelty of Test Sample')
    plt.show()
    print("Phân tích: Nếu mẫu test khác xa các mẫu train (khoảng cách lớn):")
    print("- Nếu dự đoán ĐÚNG: Mô hình có tính tổng quát hóa (generalization) tốt.")
    print("- Nếu dự đoán SAI: Mô hình gặp khó khăn với out-of-distribution (OOD) data.")
analyze_test_samples()

## 5. Model Comparison (Exp_B vs Exp_C)
Có mẫu nào cả hai đều sai không? Có mẫu nào C đúng mà B sai không?

In [ ]:
def compare_models(threshold=0.5):
    print('Comparing ResNet ASL (B) and Efficient CBAM ASL (C)...')
    targets = test_results['targets'].astype(int)
    preds_B = (test_results['preds_B'] > threshold).astype(int)
    preds_C = (test_results['preds_C'] > threshold).astype(int)
    errors_B = (preds_B != targets).astype(int)
    errors_C = (preds_C != targets).astype(int)
    both_wrong = ((errors_B == 1) & (errors_C == 1)).sum()
    B_wrong_C_right = ((errors_B == 1) & (errors_C == 0)).sum()
    C_wrong_B_right = ((errors_B == 0) & (errors_C == 1)).sum()
    both_right = ((errors_B == 0) & (errors_C == 0)).sum()
    print(f"Tổng số nhãn được đánh giá: {targets.size}")
    print(f"- Cả hai đều SAI: {both_wrong}")
    print(f"- Cả hai đều ĐÚNG: {both_right}")
    print(f"- Efficient CBAM ASL ĐÚNG nhưng ResNet ASL SAI: {B_wrong_C_right}")
    print(f"- ResNet ASL ĐÚNG nhưng Efficient CBAM ASL SAI: {C_wrong_B_right}")
compare_models()